# **1-Import of Main libraries & Load Dataset**

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import keras
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten ,Dropout,Conv2D ,MaxPooling2D , BatchNormalization , ReLU,MaxPooling2D,Flatten,AveragePooling2D
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from keras.callbacks import EarlyStopping
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/digit-recognizer/train.csv")
test = pd.read_csv("/kaggle/input/competitions/digit-recognizer/test.csv")

x_train = train.drop(columns="label")
y_train = train["label"]

x_test = test

In [ ]:
print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}")

In [ ]:
x_train = x_train.values
x_test  = x_test.values

x_train = x_train.reshape(-1, 28, 28, 1)
x_test  = x_test.reshape(-1, 28, 28, 1)

print(f"x_train shape: {x_train.shape}")
print(f"x_test shape: {x_test.shape}")

# **2-Showing All Labels of Data**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap='gray')
    plt.title(f"Label: {y_train[i]}")
    plt.axis('off')

plt.tight_layout()
plt.show()

# **3-Scalling for images**

In [ ]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# **3-Splitting Validation**

In [ ]:
x_train , x_val , y_train , y_val = train_test_split(x_train ,
                                                     y_train ,
                                                     test_size=.2 ,
                                                     stratify=y_train ,
                                                     random_state=42
                                                    )

# **4-One Hot Encoding**

In [ ]:
y_train = to_categorical(y_train, num_classes=10)
y_val   = to_categorical(y_val, num_classes=10)
print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")

# **5-CNN Scratch Model Structure**

In [ ]:
model = Sequential([
    
    Conv2D(32 , (3,3),padding="valid",input_shape=(28,28,1)),
    BatchNormalization(),
    ReLU(),
    MaxPooling2D((2,2)),
    
    Conv2D(64 , (3,3),padding="same",),
    BatchNormalization(),
    ReLU(),
    MaxPooling2D((2,2)),
    
    Conv2D(128 , (3,3),padding="valid",),
    BatchNormalization(),
    ReLU(),
    MaxPooling2D((2,2)),

    Flatten(),
    
    Dense(128),
    BatchNormalization(),
    ReLU(),
    Dropout(0.5),
    
    Dense(10 , activation="softmax")
    
])
model.summary()

# **6-Compiling & CallBacks**

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

# **7-Model Training**

In [ ]:
history=model.fit(x_train,
                  y_train,
                 epochs=50,
                 validation_data=(x_val, y_val),
                 batch_size=128,
                 callbacks=[early_stop,reduce_lr])

# **8-Evaluation and Confusion matrix**

In [ ]:
y_pred = model.predict(x_val)
predicted_labels = np.argmax(y_pred, axis=1)
true_labels      = np.argmax(y_val, axis=1)

# ===== Confusion Matrix =====
cm = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues', xticks_rotation='vertical')
plt.title("Confusion Matrix")
plt.show()

# ===== Classification Report =====
report = classification_report(true_labels, predicted_labels)
print("Classification Report:\n")
print(report)

In [ ]:
# -- Plot training vs validation accuracy --
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')

plt.xlim(0, len(history.history['accuracy']))
plt.ylim(0.8, 1)

plt.legend()
plt.title('Accuracy over epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.show()


# -- Plot training vs validation loss --
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')

plt.xlim(0, len(history.history['loss']))

plt.legend()
plt.title('Loss over epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

# **9-Test Model With Images**

In [ ]:
num_images = 10
random_indices = np.random.choice(len(x_val), size=num_images, replace=False)

X_input = x_val[random_indices]


if y_val.ndim == 2:
    y_true_labels = np.argmax(y_val[random_indices], axis=1)
else:
    y_true_labels = y_val[random_indices]


plt.figure(figsize=(20,5))
for i in range(num_images):
    plt.subplot(1, num_images, i+1)
    plt.imshow(X_input[i].reshape(28,28), cmap='gray')
    plt.title(f"True: {y_true_labels[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

for i in range(num_images):
    print(f"\nImage {i+1} (True: {y_true_labels[i]}):")
    y_pred = model.predict(X_input[i:i+1], verbose=0)
    pred_class = np.argmax(y_pred, axis=1)[0]
    emoji = "✅" if pred_class == y_true_labels[i] else "❌"
    print(f"  MLP predicts: {pred_class} {emoji}")

# **10-Model Scratch Submission**

In [ ]:
y_test_pred = model.predict(x_test, verbose=0)
predicted_labels = np.argmax(y_test_pred, axis=1)

submission = pd.DataFrame({
    "ImageId": np.arange(1,len(predicted_labels)+1),
    "Label": predicted_labels
})

# ====== حفظ CSV ======
submission.to_csv("submission_CNN.csv", index=False)
print("submission.csv saved successfully!")

# **11-LeNet-5 Structure Experment**

In [ ]:
Le_Net_model = Sequential([
    
    Conv2D(6, (5,5), activation='relu', input_shape=(28,28,1)),
    AveragePooling2D((2,2)),
    
    Conv2D(16, (5,5), activation='relu'),
    AveragePooling2D((2,2)),
    
    Flatten(),
    
    Dense(120, activation='relu'),
    Dense(84, activation='relu'),
    Dense(10, activation='softmax')
])

model.summary()

# **12-LeNet Compiling and CallBacks**

In [ ]:
Le_Net_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stop_Le_Net_model = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

reduce_lr_Le_Net_model = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

# **13-LeNet Training**

In [ ]:
history_Le_Net_model=Le_Net_model.fit(x_train,
                  y_train,
                 epochs=50,
                 validation_data=(x_val, y_val),
                 batch_size=128,
                 callbacks=[early_stop_Le_Net_model,reduce_lr_Le_Net_model])

# **14-Le Net Evaluation**

In [ ]:
y_pred = Le_Net_model.predict(x_val)
predicted_labels = np.argmax(y_pred, axis=1)
true_labels      = np.argmax(y_val, axis=1)

# ===== Confusion Matrix =====
cm = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues', xticks_rotation='vertical')
plt.title("Confusion Matrix")
plt.show()

# ===== Classification Report =====
report = classification_report(true_labels, predicted_labels)
print("Classification Report:\n")
print(report)

In [ ]:
# -- Plot training vs validation accuracy --
plt.plot(history_Le_Net_model.history['accuracy'], label='Train Accuracy')
plt.plot(history_Le_Net_model.history['val_accuracy'], label='Val Accuracy')

plt.xlim(0, len(history_Le_Net_model.history['accuracy']))
plt.ylim(0.8, 1)

plt.legend()
plt.title('Accuracy over epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.show()


# -- Plot training vs validation loss --
plt.plot(history_Le_Net_model.history['loss'], label='Train Loss')
plt.plot(history_Le_Net_model.history['val_loss'], label='Val Loss')

plt.xlim(0, len(history_Le_Net_model.history['loss']))

plt.legend()
plt.title('Loss over epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

# **15-LeNet Submission**

In [ ]:
y_test_pred = Le_Net_model.predict(x_test, verbose=0)
predicted_labels = np.argmax(y_test_pred, axis=1)

submission = pd.DataFrame({
    "ImageId": np.arange(1,len(predicted_labels)+1),
    "Label": predicted_labels
})

# ====== حفظ CSV ======
submission.to_csv("submission_LeNet-5.csv", index=False)
print("submission.csv saved successfully!")